# Report plots for all miniproject levels

This notebook runs the final controller on levels 0-4 for seeds `1`, `67`, and `777`, saves the logged data, and generates report-ready figures. X-axes are converted to physical time in seconds. Level 4 also supports video export for the dragonfly trial.

In [ ]:
from pathlib import Path
import pickle

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import trange

from flygym.compose import ActuatorType
from miniproject.simulation import MiniprojectSimulation
from submission.controller import Controller

LEVELS = [0, 1, 2, 3, 4]
SEEDS = [1, 67, 777]
STEPS_BY_LEVEL = {0: 20_000, 1: 20_000, 2: 20_000, 3: 20_000, 4: 40_000}
LOG_EVERY = 20
VIDEO_EVERY = 250
VIDEO_LEVEL = 4
VIDEO_SEED = 1
MAKE_VIDEO = True
OVERWRITE_DATA = False

DATA_DIR = Path("report_data")
FIG_DIR = Path("report_figures")
VIDEO_DIR = Path("report_videos")
for directory in (DATA_DIR, FIG_DIR, VIDEO_DIR):
    directory.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

## Helper functions

The rollout logger records fly position, distance to the banana, actual speed, commanded speed, controller drive, and dragonfly signals. Data are cached in `report_data/` so you can rerun plotting cells without rerunning every simulation.

In [ ]:
def get_body_index(sim, body_name="c_thorax"):
    body_segments = sim.fly.get_bodysegs_order()
    names = [seg.name if hasattr(seg, "name") else str(seg) for seg in body_segments]
    if body_name in names:
        return names.index(body_name)
    return 0


def get_fly_position(sim, body_idx):
    return np.asarray(sim.get_body_positions(sim.fly.name)[body_idx], dtype=float)


def get_world_frame(sim):
    sim.render_as_needed()
    frames = [frames[-1] for frames in sim.renderer.frames.values() if len(frames) > 0]
    if len(frames) == 0:
        return None
    return np.concatenate(frames, axis=-2)


def data_path(level, seed):
    return DATA_DIR / f"level{level}_seed{seed}.pkl"


def video_path(level, seed):
    return VIDEO_DIR / f"level{level}_seed{seed}.mp4"


def save_figure(fig, filename):
    out = FIG_DIR / filename
    fig.tight_layout()
    fig.savefig(out, bbox_inches="tight")
    print(f"saved {out}")
    return out


def mode_to_int(modes):
    mapping = {"normal": 0, "watch": 1, "panic_escape": 2}
    return np.array([mapping.get(str(mode), np.nan) for mode in modes], dtype=float)


def mode_background(ax, time_s, modes):
    colors = {
        "watch": (1.0, 0.82, 0.25, 0.18),
        "panic_escape": (0.95, 0.1, 0.1, 0.16),
    }
    if len(time_s) < 2:
        return
    start = 0
    modes = np.asarray(modes, dtype=object)
    for i in range(1, len(modes) + 1):
        if i == len(modes) or modes[i] != modes[start]:
            mode = str(modes[start])
            if mode in colors:
                ax.axvspan(time_s[start], time_s[i - 1], color=colors[mode], lw=0)
            start = i


def log_row(sim, controller, body_idx, previous_pos, previous_time):
    pos = get_fly_position(sim, body_idx)
    time_s = sim._curr_step * sim.timestep
    banana_xy = np.asarray(sim.world.banana_xy, dtype=float)
    distance = float(np.linalg.norm(pos[:2] - banana_xy))

    if previous_pos is None or previous_time is None or time_s <= previous_time:
        actual_speed = np.nan
    else:
        actual_speed = float(np.linalg.norm(pos[:2] - previous_pos[:2]) / (time_s - previous_time))

    current_velocity = np.asarray(controller.current_velocity, dtype=float)
    current_drive = np.asarray(controller.current_drive, dtype=float)

    dragonfly_pos = np.full(3, np.nan)
    dragonfly_distance = np.nan
    if getattr(sim, "enable_dragonfly", False):
        try:
            dragonfly_mocap_id = sim.world._get_dragonfly_mocap_id(sim)
            dragonfly_pos = sim.mj_data.mocap_pos[dragonfly_mocap_id].copy()
            dragonfly_distance = float(np.linalg.norm(pos[:3] - dragonfly_pos[:3]))
        except Exception:
            pass

    return {
        "step": sim._curr_step,
        "time_s": time_s,
        "fly_x": float(pos[0]),
        "fly_y": float(pos[1]),
        "fly_z": float(pos[2]),
        "banana_x": float(banana_xy[0]),
        "banana_y": float(banana_xy[1]),
        "distance_to_banana": distance,
        "actual_speed": actual_speed,
        "command_forward_velocity": float(current_velocity[0]),
        "command_turn_velocity": float(current_velocity[1]),
        "left_drive": float(current_drive[0]),
        "right_drive": float(current_drive[1]),
        "dragonfly_mode": str(controller.dragonfly_mode),
        "dragonfly_visible": bool(controller.dragonfly_visible),
        "dragonfly_attack": bool(controller.dragonfly_attack),
        "dragonfly_red_score": float(controller.dragonfly_red_score),
        "dragonfly_blob": float(controller.dragonfly_blob),
        "dragonfly_looming": float(controller.dragonfly_looming),
        "dragonfly_danger_score": float(controller.dragonfly_danger_score),
        "dragonfly_distance": dragonfly_distance,
        "dragonfly_is_looming": bool(getattr(sim, "_dragonfly_is_looming", False)),
        "dragonfly_is_open_loop": bool(getattr(sim, "_dragonfly_is_open_loop", False)),
    }, pos, time_s


def rows_to_arrays(rows):
    keys = rows[0].keys()
    out = {}
    for key in keys:
        values = [row[key] for row in rows]
        if isinstance(values[0], str):
            out[key] = np.asarray(values, dtype=object)
        elif isinstance(values[0], (bool, np.bool_)):
            out[key] = np.asarray(values, dtype=bool)
        else:
            out[key] = np.asarray(values, dtype=float)
    return out

## Run or load simulations

Run this cell once to generate data. It automatically loads cached `.pkl` files unless `OVERWRITE_DATA = True`.

In [ ]:
def run_trial(level, seed, make_video=False):
    path = data_path(level, seed)
    if path.exists() and not OVERWRITE_DATA:
        with path.open("rb") as f:
            return pickle.load(f)

    sim = MiniprojectSimulation(level=level, seed=seed)
    controller = Controller(sim)
    body_idx = get_body_index(sim)
    rows = []
    video_frames = []
    previous_pos = None
    previous_time = None

    max_steps = STEPS_BY_LEVEL[level]
    label = f"level {level}, seed {seed}"
    for _ in trange(max_steps, desc=label):
        joint_angles, adhesion = controller.step(sim)
        sim.set_actuator_inputs(sim.fly.name, ActuatorType.POSITION, joint_angles)
        sim.set_actuator_inputs(sim.fly.name, ActuatorType.ADHESION, adhesion)
        sim.step()

        if sim._curr_step % LOG_EVERY == 0:
            row, previous_pos, previous_time = log_row(sim, controller, body_idx, previous_pos, previous_time)
            rows.append(row)

        if make_video and sim._curr_step % VIDEO_EVERY == 0:
            frame = get_world_frame(sim)
            if frame is not None:
                video_frames.append(frame)

    arrays = rows_to_arrays(rows)
    result = {
        "level": level,
        "seed": seed,
        "timestep": sim.timestep,
        "log_every": LOG_EVERY,
        "max_steps": max_steps,
        "banana_xy": np.asarray(sim.world.banana_xy, dtype=float),
        "data": arrays,
    }

    with path.open("wb") as f:
        pickle.dump(result, f)

    if make_video and len(video_frames) > 0:
        out_video = video_path(level, seed)
        with imageio.get_writer(out_video, fps=12, codec="libx264") as writer:
            for frame in video_frames:
                writer.append_data(frame)
        result["video_path"] = str(out_video)
        with path.open("wb") as f:
            pickle.dump(result, f)
        print(f"saved {out_video}")

    return result


results = {}
for level in LEVELS:
    for seed in SEEDS:
        make_video = MAKE_VIDEO and level == VIDEO_LEVEL and seed == VIDEO_SEED
        results[(level, seed)] = run_trial(level, seed, make_video=make_video)

print("Loaded/generated", len(results), "trials")

## Plot helpers

In [ ]:
def plot_level_summary(level):
    level_results = [results[(level, seed)] for seed in SEEDS]
    fig, axs = plt.subplots(2, 2, figsize=(11, 7.2))
    axs = axs.ravel()

    for result in level_results:
        data = result["data"]
        seed = result["seed"]
        t = data["time_s"]
        axs[0].plot(t, data["distance_to_banana"], lw=1.2, alpha=0.55, label=f"seed {seed}")
        axs[1].plot(t, data["actual_speed"], lw=1.2, alpha=0.55, label=f"seed {seed}")
        axs[2].plot(data["fly_x"], data["fly_y"], lw=1.2, alpha=0.65, label=f"seed {seed}")
        axs[3].plot(t, data["command_forward_velocity"], lw=1.2, alpha=0.55, label=f"seed {seed}")

    # Mean summary across seeds on common shortest length.
    min_len = min(len(r["data"]["time_s"]) for r in level_results)
    t_common = level_results[0]["data"]["time_s"][:min_len]
    dist_stack = np.vstack([r["data"]["distance_to_banana"][:min_len] for r in level_results])
    speed_stack = np.vstack([r["data"]["actual_speed"][:min_len] for r in level_results])
    cmd_stack = np.vstack([r["data"]["command_forward_velocity"][:min_len] for r in level_results])

    for ax, stack, label in [
        (axs[0], dist_stack, "mean distance"),
        (axs[1], speed_stack, "mean actual speed"),
        (axs[3], cmd_stack, "mean commanded speed"),
    ]:
        mean = np.nanmean(stack, axis=0)
        lo = np.nanmin(stack, axis=0)
        hi = np.nanmax(stack, axis=0)
        ax.plot(t_common, mean, color="black", lw=2.2, label=label)
        ax.fill_between(t_common, lo, hi, color="black", alpha=0.12, lw=0)

    banana_xy = level_results[0]["banana_xy"]
    axs[2].scatter([banana_xy[0]], [banana_xy[1]], marker="*", s=160, c="gold", edgecolor="black", label="banana")
    axs[2].scatter([0], [0], marker="o", s=45, c="black", label="start")
    axs[2].axis("equal")

    axs[0].set_title(f"Level {level}: distance to banana")
    axs[0].set_ylabel("distance (mm)")
    axs[0].set_xlabel("time (s)")

    axs[1].set_title("Actual fly speed")
    axs[1].set_ylabel("speed (mm/s)")
    axs[1].set_xlabel("time (s)")

    axs[2].set_title("Fly trajectory")
    axs[2].set_xlabel("x position (mm)")
    axs[2].set_ylabel("y position (mm)")

    axs[3].set_title("Commanded forward velocity")
    axs[3].set_ylabel("velocity command (mm/s)")
    axs[3].set_xlabel("time (s)")

    for ax in axs:
        ax.grid(True, alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(frameon=False)

    save_figure(fig, f"level{level}_summary.png")
    plt.show()


def plot_level_individual(level):
    fig, axs = plt.subplots(len(SEEDS), 2, figsize=(11, 3.0 * len(SEEDS)), sharex="col")
    if len(SEEDS) == 1:
        axs = np.asarray([axs])

    for row, seed in enumerate(SEEDS):
        data = results[(level, seed)]["data"]
        t = data["time_s"]
        axs[row, 0].plot(t, data["distance_to_banana"], color="tab:green", lw=1.6)
        axs[row, 0].set_ylabel(f"seed {seed}\ndistance (mm)")
        axs[row, 1].plot(t, data["actual_speed"], color="tab:blue", lw=1.3, label="actual")
        axs[row, 1].plot(t, data["command_forward_velocity"], color="black", lw=1.1, ls="--", label="commanded")
        axs[row, 1].set_ylabel("speed / command")
        axs[row, 1].legend(frameon=False)

    axs[-1, 0].set_xlabel("time (s)")
    axs[-1, 1].set_xlabel("time (s)")
    axs[0, 0].set_title(f"Level {level}: distance to banana")
    axs[0, 1].set_title("Actual speed and commanded forward velocity")

    for ax in axs.ravel():
        ax.grid(True, alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)

    save_figure(fig, f"level{level}_individual_seeds.png")
    plt.show()

## Level summary figures

Each level receives a summary figure with all seeds, mean traces, and min/max spread, plus an individual-seed figure.

In [ ]:
for level in LEVELS:
    plot_level_summary(level)
    plot_level_individual(level)

## Dragonfly-specific level 4 figure

This figure shows visual predator evidence and the controller state. Colored background bands indicate `watch` and `panic_escape` periods.

In [ ]:
def plot_dragonfly_level(seed=VIDEO_SEED):
    data = results[(4, seed)]["data"]
    t = data["time_s"]
    modes = data["dragonfly_mode"]

    fig, axs = plt.subplots(4, 1, figsize=(10, 8.5), sharex=True)

    for ax in axs:
        mode_background(ax, t, modes)

    axs[0].plot(t, data["dragonfly_red_score"], color="crimson", lw=1.4, label="red score")
    axs[0].plot(t, data["dragonfly_blob"], color="black", lw=1.2, alpha=0.75, label="largest blob")
    axs[0].set_ylabel("image fraction")
    axs[0].legend(frameon=False)

    axs[1].plot(t, data["dragonfly_danger_score"], color="tab:red", lw=1.6, label="danger score")
    axs[1].set_ylabel("danger score")
    axs[1].set_ylim(-0.03, 1.03)
    axs[1].legend(frameon=False)

    axs[2].step(t, data["dragonfly_visible"].astype(float), where="post", label="visible")
    axs[2].step(t, data["dragonfly_attack"].astype(float), where="post", label="attack")
    axs[2].step(t, data["dragonfly_is_looming"].astype(float), where="post", label="sim looming", alpha=0.75)
    axs[2].step(t, data["dragonfly_is_open_loop"].astype(float), where="post", label="sim open-loop", alpha=0.75)
    axs[2].set_ylabel("state")
    axs[2].set_yticks([0, 1])
    axs[2].legend(frameon=False, ncol=2)

    axs[3].plot(t, data["command_forward_velocity"], lw=1.4, label="forward command")
    axs[3].plot(t, data["command_turn_velocity"], lw=1.4, label="turn command")
    axs[3].axhline(0, color="0.3", lw=0.8)
    axs[3].set_ylabel("velocity command")
    axs[3].set_xlabel("time (s)")
    axs[3].legend(frameon=False)

    for ax in axs:
        ax.grid(True, alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)

    save_figure(fig, f"level4_dragonfly_seed{seed}.png")
    plt.show()


plot_dragonfly_level(seed=VIDEO_SEED)

## Generated files

In [ ]:
print("Saved data files:")
for path in sorted(DATA_DIR.glob("*.pkl")):
    print(" -", path)

print("\nSaved figures:")
for path in sorted(FIG_DIR.glob("*.png")):
    print(" -", path)

print("\nSaved videos:")
for path in sorted(VIDEO_DIR.glob("*.mp4")):
    print(" -", path)